# Prostate158 QC

Visual QC for the Prostate158 pivot datasets. This notebook verifies T2/ADC/DWI images, anatomy masks, and suspicious lesion masks before nnU-Net training. The labels are research annotations, not clinical diagnoses.

In [ ]:
from pathlib import Path
import csv
import json

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np

In [ ]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').exists():
    for parent in PROJECT_ROOT.parents:
        if (parent / 'data').exists():
            PROJECT_ROOT = parent
            break

RAW_DIR = PROJECT_ROOT / 'data/raw/public/prostate158_train'
MANIFEST = PROJECT_ROOT / 'data/manifests/prostate158_manifest.csv'
DATASET502 = PROJECT_ROOT / 'data/nnunet/nnUNet_raw/Dataset502_Prostate158_Anatomy'
DATASET503 = PROJECT_ROOT / 'data/nnunet/nnUNet_raw/Dataset503_Prostate158_Lesion'
QC_DIR = PROJECT_ROOT / 'outputs/figures/qc/prostate158'
SAVE_QC_FIGURES = False

In [ ]:
def load_nifti(path):
    img = nib.load(str(path))
    return img, np.asanyarray(img.dataobj)


def choose_mask_slice(mask):
    z_counts = mask.reshape(-1, mask.shape[-1]).sum(axis=0)
    if z_counts.max() > 0:
        return int(np.argmax(z_counts))
    return mask.shape[-1] // 2


def show_overlay(image, mask, title, ax=None, alpha=0.35):
    if ax is None:
        _, ax = plt.subplots(figsize=(5, 5))
    z = choose_mask_slice(mask)
    ax.imshow(np.rot90(image[:, :, z]), cmap='gray')
    masked = np.ma.masked_where(mask[:, :, z] == 0, mask[:, :, z])
    ax.imshow(np.rot90(masked), cmap='autumn', alpha=alpha, interpolation='nearest')
    ax.set_title(f'{title} | z={z}')
    ax.axis('off')
    return ax


def read_manifest(path):
    with path.open(newline='', encoding='utf-8') as f:
        return list(csv.DictReader(f))


rows = read_manifest(MANIFEST)
positive_rows = [row for row in rows if row['lesion_present'] == 'true']
negative_rows = [row for row in rows if row['lesion_present'] == 'false']
print('manifest rows:', len(rows))
print('train rows:', sum(row['split'] == 'train' for row in rows))
print('valid rows:', sum(row['split'] == 'valid' for row in rows))
print('lesion-positive rows:', len(positive_rows))
print('lesion-negative rows:', len(negative_rows))
print('Dataset502 images:', len(list((DATASET502 / 'imagesTr').glob('*.nii.gz'))))
print('Dataset502 labels:', len(list((DATASET502 / 'labelsTr').glob('*.nii.gz'))))
print('Dataset503 images:', len(list((DATASET503 / 'imagesTr').glob('*.nii.gz'))))
print('Dataset503 labels:', len(list((DATASET503 / 'labelsTr').glob('*.nii.gz'))))

The first examples include lesion-positive and lesion-negative cases. Negative lesion cases should show empty lesion overlays in Dataset503.

In [ ]:
sample_rows = positive_rows[:3] + negative_rows[:2]
if SAVE_QC_FIGURES:
    QC_DIR.mkdir(parents=True, exist_ok=True)

for row in sample_rows:
    case_id = row['case_id']
    nnunet_id = f'prostate158_{case_id}'
    _, t2 = load_nifti(PROJECT_ROOT / row['t2w_path'])
    _, adc = load_nifti(PROJECT_ROOT / row['adc_path'])
    _, dwi = load_nifti(PROJECT_ROOT / row['dwi_path'])
    _, anatomy = load_nifti(DATASET502 / 'labelsTr' / f'{nnunet_id}.nii.gz')
    _, lesion = load_nifti(DATASET503 / 'labelsTr' / f'{nnunet_id}.nii.gz')

    print(case_id, 'anatomy values:', np.unique(anatomy), 'lesion values:', np.unique(lesion))
    assert t2.shape == adc.shape == dwi.shape == anatomy.shape == lesion.shape

    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    show_overlay(t2, anatomy, f'{case_id} T2 + anatomy', axes[0])
    show_overlay(t2, lesion, f'{case_id} T2 + lesion', axes[1])
    show_overlay(adc, lesion, f'{case_id} ADC + lesion', axes[2])
    show_overlay(dwi, lesion, f'{case_id} DWI + lesion', axes[3])
    fig.tight_layout()
    if SAVE_QC_FIGURES:
        fig.savefig(QC_DIR / f'{nnunet_id}_qc.png', dpi=150)
    plt.show()